# Generador de values Helm

Este cuaderno genera borradores de `values.yaml` por entorno para charts sencillos del curso.

In [ ]:
import copy
import yaml

base = {
    "image": {"repository": "python-api", "tag": "0.1.0", "pullPolicy": "IfNotPresent"},
    "service": {"port": 80, "targetPort": 8000},
    "resources": {
        "requests": {"cpu": "100m", "memory": "128Mi"},
        "limits": {"cpu": "300m", "memory": "256Mi"},
    },
}

overrides = {
    "dev": {"replicaCount": 1, "config": {"appEnv": "dev"}},
    "demo": {"replicaCount": 2, "config": {"appEnv": "demo"}, "ingress": {"enabled": True, "host": "demo.local"}},
    "prod": {
        "replicaCount": 3,
        "config": {"appEnv": "prod"},
        "ingress": {"enabled": True, "host": "api.example.com"},
        "resources": {
            "requests": {"cpu": "150m", "memory": "192Mi"},
            "limits": {"cpu": "500m", "memory": "512Mi"},
        },
    },
}


def deep_merge(left, right):
    result = copy.deepcopy(left)
    for key, value in right.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    return result


for env, override in overrides.items():
    values = deep_merge(base, override)
    print(f"# values-{env}.yaml")
    print(yaml.safe_dump(values, sort_keys=False))
